# UTS

## 1. Import Library

Kode !pip install Sastrawi digunakan untuk menginstal library Sastrawi ke dalam lingkungan Python, khususnya pada Google Colab. Tanda ! menunjukkan bahwa perintah tersebut dijalankan sebagai perintah sistem atau terminal, sedangkan pip install digunakan untuk memasang library Python. Library Sastrawi digunakan dalam penelitian ini untuk melakukan preprocessing teks berbahasa Indonesia, terutama pada proses stopword removal untuk menghapus kata-kata yang kurang penting dan stemming untuk mengubah kata ke bentuk dasarnya. Dengan menginstal Sastrawi terlebih dahulu, library tersebut dapat digunakan pada tahap preprocessing berita sebelum data diproses menggunakan metode TF-IDF maupun Skip-Gram.

In [ ]:
!pip install Sastrawi

Kode tersebut digunakan untuk mengimpor library yang dibutuhkan dalam tahap preprocessing data teks. import pandas as pd digunakan untuk mengolah dan memanipulasi dataset dalam bentuk tabel, misalnya membaca file CSV dan mengelola data berita. Selanjutnya, import re digunakan untuk melakukan pengolahan teks menggunakan regular expression, seperti menghapus tanda baca, angka, dan merapikan spasi. Kemudian, StopWordRemoverFactory dari Sastrawi digunakan untuk membuat proses stopword removal, yaitu menghapus kata-kata yang dianggap kurang penting dalam teks bahasa Indonesia. Sementara itu, StemmerFactory digunakan untuk membuat proses stemming, yaitu mengubah kata berimbuhan menjadi bentuk kata dasarnya. Dengan demikian, seluruh library pada kode ini dipersiapkan untuk mendukung proses pembersihan dan pengolahan teks berita sebelum digunakan dalam proses klasifikasi.

In [ ]:
import pandas as pd
import re

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

## 2. Baca dataset

Kode tersebut digunakan untuk membaca dan mengecek dataset berita yang akan digunakan dalam proses klasifikasi. pd.read_csv("dataset_detik_200_berita(1).csv") digunakan untuk membaca file dataset berformat CSV dan menyimpannya ke dalam variabel df sebagai DataFrame. Selanjutnya, len(df) digunakan untuk mengetahui jumlah data yang terdapat dalam dataset, sedangkan df.columns.tolist() digunakan untuk menampilkan nama-nama kolom yang tersedia. Terakhir, df.head() digunakan untuk menampilkan beberapa baris pertama dari dataset sehingga dapat dilakukan pengecekan awal terhadap isi dan struktur data sebelum masuk ke tahap preprocessing.

In [ ]:
df = pd.read_csv("dataset_detik_200_berita(1).csv")

print("Jumlah data:", len(df))
print("Kolom:", df.columns.tolist())

df.head()

Jumlah data: 200
Kolom: ['id', 'isi_berita', 'label']


,id,isi_berita,label
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,sport
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",sport
2,3,Nova Widianto mulai menangani tim ganda campur...,sport
3,4,Timnas basket putra Indonesia jadi rombongan p...,sport
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",sport


## 3. Cek label

Kode print(df["label"].value_counts()) digunakan untuk mengetahui jumlah data pada setiap kategori atau label yang terdapat dalam kolom label pada dataset. df["label"] digunakan untuk mengambil kolom label, kemudian value_counts() menghitung berapa banyak data yang termasuk ke dalam masing-masing kategori. Hasilnya digunakan untuk melihat distribusi kelas pada dataset, sehingga dapat diketahui apakah jumlah data antar kategori seimbang atau tidak. Dalam dataset ini, label yang digunakan adalah sport dan finance, dengan masing-masing berjumlah 100 berita, sehingga total 200 berita memiliki distribusi kelas yang seimbang.

In [ ]:
print(df["label"].value_counts())

label
sport      100
finance    100
Name: count, dtype: int64


## 4. Cek data kosong

Kode print(df[["isi_berita", "label"]].isnull().sum()) digunakan untuk memeriksa apakah terdapat data kosong (missing value) pada kolom isi_berita dan label. Bagian df[["isi_berita", "label"]] memilih kedua kolom tersebut, kemudian isnull() mendeteksi nilai yang kosong, dan sum() menghitung jumlah data kosong pada masing-masing kolom. Hasil pemeriksaan ini penting untuk memastikan bahwa teks berita dan label yang akan digunakan dalam proses klasifikasi sudah lengkap dan tidak memiliki data yang kosong.

In [ ]:
print(df[["isi_berita", "label"]].isnull().sum())

isi_berita    0
label         0
dtype: int64


## 5. Case folding

Kode tersebut digunakan untuk melakukan case folding pada data teks. Pada kode df["case_folding"] = df["isi_berita"].str.lower(), kolom isi_berita yang berisi teks berita diubah seluruh hurufnya menjadi huruf kecil menggunakan fungsi .str.lower(), kemudian hasilnya disimpan ke dalam kolom baru bernama case_folding. Selanjutnya, kode df[["isi_berita", "case_folding"]].head() digunakan untuk menampilkan lima data pertama dari kolom isi_berita dan case_folding sehingga dapat dilihat perbedaan teks sebelum dan sesudah proses case folding. Proses ini bertujuan untuk menyeragamkan bentuk huruf pada teks, sehingga kata yang sama tetapi memiliki perbedaan penggunaan huruf kapital dan huruf kecil dapat diperlakukan sebagai kata yang sama pada tahap pengolahan teks berikutnya.

In [ ]:
df["case_folding"] = df["isi_berita"].str.lower()

df[["isi_berita", "case_folding"]].head()

,isi_berita,case_folding
0,Pacuan kuda Indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
1,"Jelang Asian Games 2026, dukungan untuk 432 at...","jelang asian games 2026, dukungan untuk 432 at..."
2,Nova Widianto mulai menangani tim ganda campur...,nova widianto mulai menangani tim ganda campur...
3,Timnas basket putra Indonesia jadi rombongan p...,timnas basket putra indonesia jadi rombongan p...
4,"Bagi pelari urban, lari kini bukan lagi sekada...","bagi pelari urban, lari kini bukan lagi sekada..."


## 6. Hapus tanda baca


Kode tersebut digunakan untuk melakukan penghapusan tanda baca (punctuation removal) pada teks yang sebelumnya sudah melalui proses case folding. Pada bagian df["case_folding"].apply(...), setiap teks dalam kolom case_folding diproses satu per satu. Fungsi lambda x digunakan untuk mengambil setiap teks, kemudian re.sub(r"[^\w\s]", " ", x) dari library re mengganti semua karakter yang bukan huruf, angka, underscore, atau spasi dengan spasi. Dengan demikian, tanda baca seperti titik (.), koma (,), tanda seru (!), tanda tanya (?), dan tanda baca lainnya akan dihapus. Hasil proses tersebut kemudian disimpan ke dalam kolom baru bernama tanpa_tanda_baca, sehingga teks menjadi lebih bersih dan siap digunakan untuk tahap preprocessing berikutnya.

In [ ]:
df["tanpa_tanda_baca"] = df["case_folding"].apply(
    lambda x: re.sub(r"[^\w\s]", " ", x)
)

## 7. Hapus angka

Kode tersebut digunakan untuk melakukan penghapusan angka (number removal) dari teks yang sudah melalui proses penghapusan tanda baca. Pada bagian df["tanpa_tanda_baca"].apply(...), setiap teks dalam kolom tanpa_tanda_baca diproses satu per satu. Fungsi re.sub(r"\d+", " ", x) digunakan untuk mencari angka yang terdapat dalam teks dan menggantinya dengan spasi. Pola \d+ menunjukkan satu atau lebih angka yang berurutan, sehingga angka seperti 2024, 10, atau 123 akan dihapus. Hasil dari proses tersebut kemudian disimpan dalam kolom baru tanpa_angka. Dengan demikian, teks menjadi lebih bersih karena tidak lagi mengandung angka dan dapat dilanjutkan ke tahap preprocessing berikutnya.

In [ ]:
df["tanpa_angka"] = df["tanpa_tanda_baca"].apply(
    lambda x: re.sub(r"\d+", " ", x)
)

## 8. Rapikan spasi

Kode tersebut digunakan untuk melakukan pembersihan spasi berlebih (whitespace normalization) pada teks yang sudah melalui proses penghapusan angka. Pada bagian df["tanpa_angka"].apply(...), setiap teks dalam kolom tanpa_angka diproses satu per satu. Fungsi re.sub(r"\s+", " ", x) digunakan untuk mengganti spasi atau karakter whitespace yang berulang menjadi satu spasi saja. Selanjutnya, fungsi .strip() digunakan untuk menghapus spasi yang terdapat di awal dan akhir teks. Hasil dari proses tersebut disimpan dalam kolom baru bernama clean_text. Dengan demikian, teks menjadi lebih rapi dan memiliki jarak antar kata yang seragam sehingga siap digunakan untuk tahap preprocessing teks selanjutnya.

In [ ]:
df["clean_text"] = df["tanpa_angka"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

In [ ]:
df[["isi_berita", "clean_text"]].head(3)

,isi_berita,clean_text
0,Pacuan kuda Indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
1,"Jelang Asian Games 2026, dukungan untuk 432 at...",jelang asian games dukungan untuk atlet indone...
2,Nova Widianto mulai menangani tim ganda campur...,nova widianto mulai menangani tim ganda campur...


## 9. Tokenisasi

Kode tersebut digunakan untuk melakukan tokenisasi, yaitu memecah teks menjadi bagian-bagian yang lebih kecil yang disebut token, biasanya berupa kata. Pada kode df["clean_text"].apply(lambda x: x.split()), setiap teks pada kolom clean_text diproses menggunakan fungsi .split() untuk memisahkan teks berdasarkan spasi. Hasil pemisahan tersebut kemudian disimpan dalam kolom baru bernama tokens. Selanjutnya, kode df[["clean_text", "tokens"]].head(3) digunakan untuk menampilkan tiga data pertama dari kolom clean_text dan tokens, sehingga dapat dilihat perubahan teks sebelum dan sesudah proses tokenisasi.

In [ ]:
df["tokens"] = df["clean_text"].apply(lambda x: x.split())

df[["clean_text", "tokens"]].head(3)

,clean_text,tokens
0,pacuan kuda indonesia mulai diarahkan menuju p...,"[pacuan, kuda, indonesia, mulai, diarahkan, me..."
1,jelang asian games dukungan untuk atlet indone...,"[jelang, asian, games, dukungan, untuk, atlet,..."
2,nova widianto mulai menangani tim ganda campur...,"[nova, widianto, mulai, menangani, tim, ganda,..."


## 10. Stopword removal

Kode tersebut digunakan untuk melakukan penghapusan stopword, yaitu kata-kata yang dianggap kurang memiliki makna penting dalam proses klasifikasi teks, seperti yang, dan, di, ke, dari, dan sebagainya. Pada bagian StopWordRemoverFactory(), dibuat objek untuk menyediakan daftar stopword bahasa Indonesia, kemudian create_stop_word_remover() digunakan untuk membuat alat yang bertugas menghapus stopword. Selanjutnya, setiap teks pada kolom clean_text diproses menggunakan stopword_remover.remove(x) dan hasilnya disimpan dalam kolom baru text_no_stopword. Terakhir, df[["clean_text", "text_no_stopword"]].head(3) digunakan untuk menampilkan tiga data pertama sebelum dan sesudah penghapusan stopword, sehingga dapat dilihat kata-kata yang tidak diperlukan telah dihilangkan dari teks.

In [ ]:
stop_factory = StopWordRemoverFactory()
stopword_remover = stop_factory.create_stop_word_remover()

df["text_no_stopword"] = df["clean_text"].apply(
    lambda x: stopword_remover.remove(x)
)

df[["clean_text", "text_no_stopword"]].head(3)

,clean_text,text_no_stopword
0,pacuan kuda indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
1,jelang asian games dukungan untuk atlet indone...,jelang asian games dukungan atlet indonesia te...
2,nova widianto mulai menangani tim ganda campur...,nova widianto mulai menangani tim ganda campur...


## 11. Stemming

Kode tersebut digunakan untuk melakukan stemming, yaitu mengubah kata-kata dalam teks menjadi bentuk dasarnya. Pada bagian StemmerFactory(), dibuat objek yang menyediakan proses stemming bahasa Indonesia, kemudian create_stemmer() digunakan untuk membuat alat stemmer yang akan mengubah kata ke bentuk dasarnya. Selanjutnya, setiap teks pada kolom text_no_stopword diproses menggunakan stemmer.stem(x), lalu hasilnya disimpan dalam kolom baru bernama text_stemmed. Sebagai contoh, kata berjalan dapat diubah menjadi jalan, membaca menjadi baca, dan penjualan menjadi jual. Terakhir, kode df[["clean_text", "text_stemmed"]].head(3) digunakan untuk menampilkan tiga data pertama dari teks sebelum dan sesudah stemming, sehingga dapat dilihat perubahan kata yang terjadi setelah proses stemming.

In [ ]:
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

df["text_stemmed"] = df["text_no_stopword"].apply(
    lambda x: stemmer.stem(x)
)

df[["clean_text", "text_stemmed"]].head(3)

,clean_text,text_stemmed
0,pacuan kuda indonesia mulai diarahkan menuju p...,pacu kuda indonesia mulai arah tuju panggung d...
1,jelang asian games dukungan untuk atlet indone...,jelang asi games dukung atlet indonesia terus ...
2,nova widianto mulai menangani tim ganda campur...,nova widianto mulai tangan tim ganda campur pe...


Kode tersebut digunakan untuk memisahkan teks yang sudah melalui proses stemming menjadi token-token berupa kata. Pada bagian df["text_stemmed"].apply(lambda x: x.split()), setiap teks pada kolom text_stemmed dipisahkan berdasarkan spasi menggunakan fungsi .split(), kemudian hasilnya disimpan dalam kolom baru bernama tokens_stemmed. Dengan demikian, teks yang sebelumnya berbentuk kalimat akan berubah menjadi kumpulan kata dalam bentuk list. Selanjutnya, kode df[["text_stemmed", "tokens_stemmed", "label"]].head(5) digunakan untuk menampilkan lima data pertama yang terdiri dari teks hasil stemming, token-token hasil stemming, dan label atau kategori dari masing-masing data. Hal ini bertujuan untuk melihat hasil akhir tokenisasi setelah proses stemming sekaligus memastikan bahwa setiap teks masih memiliki label yang sesuai untuk digunakan dalam proses klasifikasi.

In [ ]:
df["tokens_stemmed"] = df["text_stemmed"].apply(
    lambda x: x.split()
)

df[["text_stemmed", "tokens_stemmed", "label"]].head(5)

,text_stemmed,tokens_stemmed,label
0,pacu kuda indonesia mulai arah tuju panggung d...,"[pacu, kuda, indonesia, mulai, arah, tuju, pan...",sport
1,jelang asi games dukung atlet indonesia terus ...,"[jelang, asi, games, dukung, atlet, indonesia,...",sport
2,nova widianto mulai tangan tim ganda campur pe...,"[nova, widianto, mulai, tangan, tim, ganda, ca...",sport
3,timnas basket putra indonesia jadi rombong per...,"[timnas, basket, putra, indonesia, jadi, rombo...",sport
4,lari urban lari kini bukan sekadar kejar catat...,"[lari, urban, lari, kini, bukan, sekadar, keja...",sport


## 12. Import library untuk TF-IDF dan Naive Bayes

Kode tersebut digunakan untuk mempersiapkan library yang dibutuhkan dalam proses klasifikasi teks menggunakan algoritma Naive Bayes. train_test_split digunakan untuk membagi dataset menjadi data latih (training data) dan data uji (testing data). TfidfVectorizer digunakan untuk mengubah teks menjadi representasi numerik berdasarkan bobot TF-IDF, sehingga teks dapat diproses oleh algoritma machine learning. MultinomialNB merupakan algoritma Multinomial Naive Bayes yang digunakan untuk melakukan klasifikasi teks berdasarkan pola kata yang terdapat dalam data. Sementara itu, accuracy_score, classification_report, dan confusion_matrix digunakan untuk mengevaluasi kinerja model. accuracy_score digunakan untuk menghitung tingkat akurasi, classification_report menampilkan metrik seperti precision, recall, dan F1-score, sedangkan confusion_matrix menunjukkan jumlah prediksi yang benar dan salah pada setiap kelas.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 13. Siapkan X dan y hasil preprocessing

Kode tersebut digunakan untuk menentukan data yang akan digunakan sebagai fitur dan target dalam proses klasifikasi. Pada bagian X = df["text_stemmed"], kolom text_stemmed ditetapkan sebagai X atau data fitur (input) yang berisi teks berita yang sudah melalui proses preprocessing dan stemming. Sementara itu, y = df["label"] menetapkan kolom label sebagai y atau target (output) yang berisi kategori atau kelas yang akan diprediksi oleh model. Selanjutnya, print("Jumlah data:", len(X)) digunakan untuk menampilkan jumlah data teks yang akan digunakan, sedangkan print("Jumlah label:", len(y)) digunakan untuk menampilkan jumlah label. Jumlah data dan label seharusnya sama karena setiap teks harus memiliki satu label yang sesuai untuk proses klasifikasi.

In [ ]:
X = df["text_stemmed"]
y = df["label"]

print("Jumlah data:", len(X))
print("Jumlah label:", len(y))

Jumlah data: 200
Jumlah label: 200


## 14. Bagi data training dan testing

Kode tersebut digunakan untuk membagi dataset menjadi data training dan data testing sebelum model klasifikasi Naive Bayes dilatih. Fungsi train_test_split() membagi X yang berisi teks dan y yang berisi label menjadi empat bagian, yaitu X_train dan y_train sebagai data training, serta X_test dan y_test sebagai data testing. Parameter test_size=0.2 berarti 20% data digunakan sebagai data testing, sedangkan 80% sisanya digunakan sebagai data training. Parameter random_state=42 digunakan agar pembagian data selalu menghasilkan pembagian yang sama ketika kode dijalankan kembali. Sementara itu, stratify=y digunakan agar proporsi setiap kelas pada data training dan testing tetap seimbang atau mendekati proporsi pada dataset awal. Terakhir, print() digunakan untuk menampilkan jumlah data yang masuk ke bagian training dan testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data training:", len(X_train))
print("Data testing:", len(X_test))

Data training: 160
Data testing: 40


## 15. Buat TF-IDF

Kode tersebut digunakan untuk mengubah teks menjadi bentuk numerik menggunakan metode TF-IDF (Term Frequency–Inverse Document Frequency) agar teks dapat diproses oleh algoritma Naive Bayes. Pada TfidfVectorizer(), dibuat objek untuk melakukan proses pembobotan kata. Kemudian, tfidf.fit_transform(X_train) digunakan untuk mempelajari kosakata dan bobot TF-IDF dari data training, lalu mengubah teks training menjadi matriks numerik yang disimpan dalam X_train_tfidf. Selanjutnya, tfidf.transform(X_test) digunakan untuk mengubah data testing ke dalam bentuk TF-IDF menggunakan kosakata dan bobot yang sudah dipelajari dari data training. Pemisahan proses fit_transform pada data training dan transform pada data testing penting agar informasi dari data testing tidak digunakan saat proses pembelajaran model.

In [ ]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Kode tersebut digunakan untuk menampilkan ukuran matriks TF-IDF yang telah dihasilkan dari data training dan data testing. Pada X_train_tfidf.shape, program menampilkan jumlah baris dan kolom pada matriks TF-IDF training, sedangkan X_test_tfidf.shape menampilkan ukuran matriks TF-IDF testing. Jumlah baris menunjukkan jumlah dokumen atau data, sementara jumlah kolom menunjukkan jumlah fitur atau kata unik (kosakata) yang digunakan dalam proses TF-IDF. Dengan demikian, kode ini bertujuan untuk memastikan bahwa proses transformasi teks menjadi matriks numerik telah dilakukan dengan benar sebelum data digunakan untuk melatih model Naive Bayes.

In [ ]:
print("Ukuran matrix TF-IDF training:", X_train_tfidf.shape)
print("Ukuran matrix TF-IDF testing:", X_test_tfidf.shape)

Ukuran matrix TF-IDF training: (160, 4494)
Ukuran matrix TF-IDF testing: (40, 4494)


Kode tersebut digunakan untuk melakukan normalisasi nilai TF-IDF menggunakan Min-Max Scaling agar nilai setiap fitur berada dalam rentang yang seragam. MinMaxScaler() membuat objek scaler yang digunakan untuk mengubah rentang nilai fitur. Pada X_train_tfidf.toarray(), matriks TF-IDF training yang sebelumnya berbentuk sparse matrix diubah menjadi array biasa agar dapat diproses oleh MinMaxScaler. Selanjutnya, fit_transform() pada X_train_tfidf digunakan untuk mempelajari nilai minimum dan maksimum dari data training sekaligus melakukan normalisasi, kemudian hasilnya disimpan dalam X_train_tfidf_scaled. Sementara itu, transform() digunakan untuk melakukan normalisasi pada data testing berdasarkan skala yang telah dipelajari dari data training, sehingga tidak terjadi kebocoran informasi dari data testing. Terakhir, min() dan max() digunakan untuk memeriksa nilai minimum dan maksimum hasil normalisasi. Dengan proses ini, nilai fitur TF-IDF berada pada rentang 0 sampai 1.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_tfidf = MinMaxScaler()

X_train_tfidf_scaled = scaler_tfidf.fit_transform(
    X_train_tfidf.toarray()
)

X_test_tfidf_scaled = scaler_tfidf.transform(
    X_test_tfidf.toarray()
)

print("Nilai minimum:", X_train_tfidf_scaled.min())
print("Nilai maksimum:", X_train_tfidf_scaled.max())

Nilai minimum: 0.0
Nilai maksimum: 1.0


## 16. Buat model Naive Bayes

Kode tersebut digunakan untuk membuat dan melatih model klasifikasi Multinomial Naive Bayes menggunakan data TF-IDF yang telah dinormalisasi. Pada MultinomialNB(), dibuat model Naive Bayes dengan jenis Multinomial, yang sesuai digunakan untuk klasifikasi berbasis fitur teks. Selanjutnya, nb_tfidf.fit(X_train_tfidf_scaled, y_train) digunakan untuk melatih model menggunakan data training TF-IDF yang sudah melalui Min-Max Scaling sebagai fitur dan y_train sebagai label atau target klasifikasi. Selama proses ini, model mempelajari hubungan antara fitur kata dan kategori berita berdasarkan data training. Setelah proses pelatihan selesai, print() digunakan untuk memberikan informasi bahwa model MultinomialNB telah berhasil dilatih dan siap digunakan untuk melakukan prediksi pada data testing.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_tfidf = MultinomialNB()

nb_tfidf.fit(
    X_train_tfidf_scaled,
    y_train
)

print("Model MultinomialNB berhasil dilatih.")

Model MultinomialNB berhasil dilatih.


Kode tersebut digunakan untuk menampilkan informasi mengenai model dan data yang digunakan dalam proses pelatihan Multinomial Naive Bayes. Pada print("Model:", nb_tfidf), program menampilkan jenis model yang digunakan, yaitu MultinomialNB. Kemudian, X_train_tfidf.shape[0] digunakan untuk mengetahui jumlah data training, yaitu jumlah dokumen atau berita yang digunakan untuk melatih model. Sementara itu, X_train_tfidf.shape[1] digunakan untuk mengetahui jumlah fitur, yaitu jumlah kata atau kosakata unik yang direpresentasikan dalam matriks TF-IDF. Dengan demikian, kode ini digunakan untuk memastikan dan memberikan gambaran mengenai model, jumlah data training, serta jumlah fitur yang digunakan dalam proses klasifikasi.

In [ ]:
print("Model:", nb_tfidf)
print("Jumlah data training:", X_train_tfidf.shape[0])
print("Jumlah fitur:", X_train_tfidf.shape[1])

Model: MultinomialNB()
Jumlah data training: 160
Jumlah fitur: 4494


## 17. Prediksi

Kode tersebut digunakan untuk melakukan prediksi terhadap data testing menggunakan model Multinomial Naive Bayes yang telah dilatih sebelumnya. Pada nb_tfidf.predict(X_test_tfidf_scaled), model nb_tfidf menerima data testing yang sudah melalui proses TF-IDF dan normalisasi Min-Max, kemudian memprediksi label atau kategori untuk setiap data. Hasil prediksi tersebut disimpan dalam variabel y_pred_tfidf. Selanjutnya, perintah print() digunakan untuk menampilkan seluruh hasil prediksi sehingga dapat dilihat kategori yang diprediksi oleh model untuk masing-masing data testing. Hasil prediksi ini nantinya dapat dibandingkan dengan y_test, yaitu label sebenarnya dari data testing, untuk mengukur kinerja model menggunakan akurasi, precision, recall, F1-score, dan confusion matrix.

In [ ]:
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf_scaled)

print("Hasil prediksi:")
print(y_pred_tfidf)

Hasil prediksi:
['finance' 'sport' 'sport' 'sport' 'finance' 'sport' 'finance' 'sport'
 'sport' 'finance' 'sport' 'sport' 'finance' 'sport' 'finance' 'finance'
 'sport' 'finance' 'finance' 'sport' 'finance' 'sport' 'sport' 'sport'
 'sport' 'sport' 'sport' 'finance' 'sport' 'finance' 'sport' 'finance'
 'finance' 'finance' 'finance' 'finance' 'finance' 'finance' 'sport'
 'finance']


## 18. Evaluasi

Kode tersebut digunakan untuk mengevaluasi kinerja model Multinomial Naive Bayes dengan membandingkan label sebenarnya (y_test) dengan hasil prediksi model (y_pred_tfidf). Pada accuracy_score(y_test, y_pred_tfidf), program menghitung akurasi, yaitu persentase data testing yang berhasil diklasifikasikan dengan benar oleh model. Sementara itu, classification_report(y_test, y_pred_tfidf) menampilkan beberapa metrik evaluasi, yaitu precision, recall, F1-score, dan support untuk setiap kelas. Precision menunjukkan ketepatan model dalam memprediksi suatu kelas, recall menunjukkan kemampuan model menemukan seluruh data yang sebenarnya termasuk dalam kelas tersebut, sedangkan F1-score merupakan rata-rata harmonis antara precision dan recall. Support menunjukkan jumlah data sebenarnya pada masing-masing kelas. Dengan demikian, kode ini digunakan untuk mengetahui seberapa baik model Multinomial Naive Bayes melakukan klasifikasi terhadap data testing.

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print(classification_report(y_test, y_pred_tfidf))

Accuracy: 1.0
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00        20
       sport       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



## 19. Install/import Word2Vec

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 59.2 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

## 20. Siapkan data Skip-Gram

Kode tersebut digunakan untuk mengambil data hasil tokenisasi dan memeriksa struktur data sebelum digunakan pada proses berikutnya. Pada df["tokens_stemmed"].tolist(), kolom tokens_stemmed yang berisi kumpulan kata hasil stemming diubah menjadi sebuah list Python dan disimpan dalam variabel sentences. Selanjutnya, len(sentences) digunakan untuk mengetahui jumlah dokumen atau data teks yang terdapat dalam list tersebut. Sementara itu, sentences[0][:10] digunakan untuk menampilkan 10 token pertama dari dokumen pertama, sehingga dapat diperiksa apakah hasil tokenisasi dan stemming sudah sesuai. Jadi, kode ini pada dasarnya digunakan untuk menyiapkan serta mengecek data teks yang sudah dalam bentuk token sebelum masuk ke tahap pemodelan atau analisis selanjutnya.

In [ ]:
sentences = df["tokens_stemmed"].tolist()

print("Jumlah dokumen:", len(sentences))
print("Contoh:", sentences[0][:10])

Jumlah dokumen: 200
Contoh: ['pacu', 'kuda', 'indonesia', 'mulai', 'arah', 'tuju', 'panggung', 'dunia', 'lalu', 'indonesia']


## 21. Buat model Skip-Gram

Kode tersebut digunakan untuk membuat model Word2Vec dengan metode Skip-Gram menggunakan data training saja. Pertama, np.arange(len(df)) membuat indeks untuk seluruh data, kemudian train_test_split() membagi indeks menjadi data training dan testing dengan proporsi 80% training dan 20% testing. Parameter stratify=df["label"] digunakan agar proporsi setiap kelas tetap terjaga pada kedua bagian data. Selanjutnya, df["tokens_stemmed"].iloc[train_idx].tolist() mengambil token hasil stemming hanya dari data training, sehingga data testing tidak ikut digunakan dalam pembentukan Word2Vec dan tidak terjadi kebocoran data. Setelah itu, Word2Vec() digunakan untuk membangun model Skip-Gram, yaitu metode Word2Vec yang mempelajari hubungan antar kata dengan memprediksi kata-kata di sekitar suatu kata. Parameter vector_size=100 menentukan bahwa setiap kata akan direpresentasikan sebagai vektor dengan 100 dimensi, window=5 menentukan jumlah kata di sekitar kata target yang digunakan sebagai konteks, min_count=1 berarti kata yang muncul minimal satu kali tetap dimasukkan ke vocabulary, workers=4 menentukan jumlah thread yang digunakan, sg=1 memilih metode Skip-Gram, sedangkan epochs=20 menentukan model dilatih sebanyak 20 kali terhadap data training. Terakhir, kode menampilkan jumlah kata yang masuk ke vocabulary dan ukuran vektor yang dihasilkan.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec

In [ ]:
# Siapkan index data training dan testing
indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# Ambil token hanya dari data training
train_tokens = df["tokens_stemmed"].iloc[train_idx].tolist()

# Buat model Skip-Gram dari data training saja
skipgram_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=20
)

print("Model Skip-Gram berhasil dibuat.")
print("Jumlah vocabulary:", len(skipgram_model.wv))
print("Ukuran vector:", skipgram_model.wv.vector_size)

Model Skip-Gram berhasil dibuat.
Jumlah vocabulary: 4514
Ukuran vector: 100


## 22. Cek ukuran vocabulary

Kode tersebut digunakan untuk menampilkan informasi mengenai hasil model Skip-Gram yang telah dibuat. Pada len(skipgram_model.wv), program menghitung jumlah kata yang terdapat dalam vocabulary, yaitu kumpulan kata yang dipelajari oleh model dari data training. Sementara itu, skipgram_model.wv.vector_size digunakan untuk mengetahui jumlah dimensi vektor yang digunakan untuk merepresentasikan setiap kata. Dalam kode sebelumnya, parameter vector_size=100, sehingga setiap kata direpresentasikan dalam bentuk vektor dengan 100 dimensi. Dengan demikian, kode ini digunakan untuk mengetahui berapa banyak kata yang dipelajari model dan berapa dimensi representasi vektor setiap kata.

In [ ]:
print("Jumlah vocabulary:", len(skipgram_model.wv))
print("Ukuran vector:", skipgram_model.wv.vector_size)

Jumlah vocabulary: 4514
Ukuran vector: 100


## 23. Ubah setiap berita menjadi satu vektor

Kode tersebut digunakan untuk membuat **representasi vektor untuk setiap dokumen berdasarkan vektor kata dari model Skip-Gram**. Fungsi `document_vector(tokens, model)` menerima `tokens` berupa kumpulan kata dalam sebuah dokumen dan `model` berupa model Word2Vec Skip-Gram yang telah dilatih. Variabel `vectors` digunakan untuk menyimpan vektor dari setiap kata yang terdapat dalam vocabulary model. Perulangan `for word in tokens` memeriksa setiap kata dalam dokumen, sedangkan `if word in model.wv` memastikan bahwa hanya kata yang dikenal oleh model yang digunakan. Jika kata ditemukan, vektor kata tersebut ditambahkan ke dalam `vectors`. Apabila tidak ada satu pun kata yang memiliki vektor, fungsi mengembalikan **vektor nol** dengan ukuran sesuai `model.vector_size`. Jika terdapat kata yang dikenal, `np.mean(vectors, axis=0)` menghitung **rata-rata seluruh vektor kata dalam dokumen**, sehingga setiap dokumen direpresentasikan menjadi satu vektor dengan ukuran yang sama, yaitu **100 dimensi** pada model Skip-Gram sebelumnya.


In [ ]:
import numpy as np

def document_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

Kode tersebut digunakan untuk **mengubah seluruh dokumen berita menjadi representasi numerik berdasarkan model Word2Vec Skip-Gram yang telah dilatih sebelumnya**. Pada `df["tokens_stemmed"].tolist()`, seluruh data pada kolom `tokens_stemmed` diubah menjadi list dan disimpan dalam variabel `all_tokens`. Selanjutnya, setiap dokumen diproses menggunakan fungsi `document_vector()`, yang menghitung rata-rata vektor dari kata-kata dalam dokumen berdasarkan model Skip-Gram. Hasil dari setiap dokumen kemudian dikumpulkan dan diubah menjadi array NumPy menggunakan `np.array()`, lalu disimpan dalam variabel `X_skipgram`. Dengan demikian, setiap berita direpresentasikan dalam bentuk **vektor numerik 100 dimensi** sesuai dengan `vector_size=100` pada model Skip-Gram. Terakhir, `X_skipgram.shape` digunakan untuk menampilkan ukuran matriks yang dihasilkan, yaitu **jumlah dokumen × jumlah dimensi vektor**.


In [ ]:
# Ubah seluruh berita menjadi vektor
# Word2Vec sudah dilatih hanya menggunakan data training
all_tokens = df["tokens_stemmed"].tolist()

X_skipgram = np.array([
    document_vector(tokens, skipgram_model)
    for tokens in all_tokens
])

print("Ukuran matriks Skip-Gram:", X_skipgram.shape)

Ukuran matriks Skip-Gram: (200, 100)


## 24. Bagi data training dan testing

Kode tersebut digunakan untuk **membagi data hasil representasi Word2Vec Skip-Gram menjadi data training dan data testing**. Pada `X_train_skipgram = X_skipgram[train_idx]`, data vektor dokumen yang memiliki indeks `train_idx` diambil sebagai **data training**, sedangkan `X_test_skipgram = X_skipgram[test_idx]` mengambil data dengan indeks `test_idx` sebagai **data testing**. Selanjutnya, `y_train_skipgram` dan `y_test_skipgram` mengambil **label** yang sesuai dengan data training dan testing tersebut. Pembagian ini menggunakan indeks yang sebelumnya sudah dibuat dengan `train_test_split()`, sehingga pembagian data dan label tetap sesuai. Terakhir, `shape` digunakan untuk menampilkan ukuran masing-masing data, yaitu **jumlah dokumen × jumlah dimensi vektor**. Dengan vector size sebesar 100, setiap dokumen direpresentasikan menggunakan 100 fitur numerik.


In [ ]:
# Bagi data training dan testing

X_train_skipgram = X_skipgram[train_idx]
X_test_skipgram = X_skipgram[test_idx]

y_train_skipgram = df["label"].iloc[train_idx]
y_test_skipgram = df["label"].iloc[test_idx]

print("Ukuran data training:", X_train_skipgram.shape)
print("Ukuran data testing:", X_test_skipgram.shape)

Ukuran data training: (160, 100)
Ukuran data testing: (40, 100)


## 25. Naive Bayes

Kode tersebut digunakan untuk **membuat dan melatih model Gaussian Naive Bayes** menggunakan data berita yang telah diubah menjadi vektor numerik dengan metode **Word2Vec Skip-Gram**. Pada `GaussianNB()`, dibuat model klasifikasi Gaussian Naive Bayes yang sesuai untuk menangani fitur numerik kontinu seperti nilai-nilai pada vektor Word2Vec. Selanjutnya, `nb_skipgram.fit(X_train_skipgram, y_train_skipgram)` digunakan untuk melatih model menggunakan **vektor dokumen dari data training** sebagai fitur dan `y_train_skipgram` sebagai label atau kategori berita. Selama proses pelatihan, model mempelajari pola distribusi nilai fitur pada masing-masing kelas untuk digunakan dalam melakukan klasifikasi. Setelah proses pelatihan selesai, `print()` digunakan untuk memastikan bahwa model Gaussian Naive Bayes telah berhasil dilatih dan siap digunakan untuk memprediksi data testing.



In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_skipgram = GaussianNB()

nb_skipgram.fit(
    X_train_skipgram,
    y_train_skipgram
)

print("Model Gaussian Naive Bayes berhasil dilatih.")

Model Gaussian Naive Bayes berhasil dilatih.


## 26. Prediksi

Kode tersebut digunakan untuk **melakukan prediksi terhadap data testing menggunakan model Gaussian Naive Bayes** yang telah dilatih sebelumnya. Pada `nb_skipgram.predict(X_test_skipgram)`, model `nb_skipgram` menerima data testing yang berupa vektor hasil representasi **Word2Vec Skip-Gram**, kemudian menentukan label atau kategori yang diprediksi untuk setiap berita. Hasil prediksi tersebut disimpan dalam variabel **`y_pred_skipgram`**. Selanjutnya, perintah `print()` digunakan untuk menampilkan seluruh hasil prediksi dari data testing. Hasil ini nantinya dapat dibandingkan dengan **`y_test_skipgram`**, yaitu label sebenarnya dari data testing, untuk mengetahui seberapa baik model melakukan klasifikasi.


In [ ]:
# Prediksi data testing
y_pred_skipgram = nb_skipgram.predict(X_test_skipgram)

print("Hasil prediksi:")
print(y_pred_skipgram)

Hasil prediksi:
['finance' 'sport' 'sport' 'sport' 'finance' 'sport' 'finance' 'sport'
 'sport' 'finance' 'sport' 'sport' 'finance' 'sport' 'finance' 'finance'
 'sport' 'finance' 'finance' 'sport' 'finance' 'sport' 'sport' 'sport'
 'sport' 'sport' 'sport' 'finance' 'sport' 'finance' 'sport' 'finance'
 'finance' 'finance' 'finance' 'finance' 'finance' 'finance' 'sport'
 'finance']


## 27. Evaluasi

Kode tersebut digunakan untuk **mengevaluasi kinerja model Gaussian Naive Bayes yang menggunakan representasi Word2Vec Skip-Gram**. Pada `accuracy_score(y_test_skipgram, y_pred_skipgram)`, program membandingkan label sebenarnya pada data testing (`y_test_skipgram`) dengan hasil prediksi model (`y_pred_skipgram`) untuk menghitung **nilai akurasi**, yaitu persentase data yang berhasil diklasifikasikan dengan benar. Hasil akurasi kemudian disimpan dalam variabel `accuracy_skipgram` dan ditampilkan. Selanjutnya, `classification_report()` digunakan untuk menampilkan **precision, recall, F1-score, dan support** pada setiap kelas. Dengan demikian, kode ini digunakan untuk mengetahui seberapa baik kombinasi **Word2Vec Skip-Gram + Gaussian Naive Bayes** dalam melakukan klasifikasi teks berdasarkan data testing.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_skipgram = accuracy_score(
    y_test_skipgram,
    y_pred_skipgram
)

print("Accuracy Skip-Gram + GaussianNB:", accuracy_skipgram)

print("\nClassification Report:")
print(
    classification_report(
        y_test_skipgram,
        y_pred_skipgram
    )
)

Accuracy Skip-Gram + GaussianNB: 1.0

Classification Report:
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00        20
       sport       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



## 28. Buat tabel perbandingan

Kode tersebut digunakan untuk **membuat tabel perbandingan kinerja dari dua metode klasifikasi teks**, yaitu **TF-IDF + Multinomial Naive Bayes** dan **Skip-Gram + Gaussian Naive Bayes**. Pada `pd.DataFrame()`, dibuat sebuah DataFrame bernama `perbandingan` yang memiliki dua kolom, yaitu `Metode` untuk menunjukkan metode yang digunakan dan `Accuracy` untuk menyimpan nilai akurasi masing-masing metode. Nilai `accuracy_tfidf` merupakan akurasi dari metode **TF-IDF + MultinomialNB**, sedangkan `accuracy_skipgram` merupakan akurasi dari metode **Skip-Gram + GaussianNB**. Selanjutnya, `print(perbandingan)` digunakan untuk menampilkan tabel tersebut sehingga dapat dilihat dan dibandingkan metode mana yang menghasilkan **akurasi lebih tinggi** dalam melakukan klasifikasi teks.


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.metrics import accuracy_score

# Accuracy TF-IDF + MultinomialNB
accuracy_tfidf = accuracy_score(
    y_test,
    y_pred_tfidf
)

# Accuracy Skip-Gram + GaussianNB
accuracy_skipgram = accuracy_score(
    y_test_skipgram,
    y_pred_skipgram
)

print("Accuracy TF-IDF + MultinomialNB :", accuracy_tfidf)
print("Accuracy Skip-Gram + GaussianNB :", accuracy_skipgram)

Accuracy TF-IDF + MultinomialNB : 1.0
Accuracy Skip-Gram + GaussianNB : 1.0


In [ ]:
import pandas as pd

perbandingan = pd.DataFrame({
    "Metode": [
        "TF-IDF + MultinomialNB",
        "Skip-Gram + GaussianNB"
    ],
    "Accuracy": [
        accuracy_tfidf,
        accuracy_skipgram
    ]
})

perbandingan["Accuracy"] = (
    perbandingan["Accuracy"] * 100
).round(2)

print(perbandingan)

                   Metode  Accuracy
0  TF-IDF + MultinomialNB     100.0
1  Skip-Gram + GaussianNB     100.0


Kode tersebut digunakan untuk **menampilkan informasi ukuran data dan karakteristik fitur** dari dua metode yang digunakan, yaitu **TF-IDF** dan **Skip-Gram**. `X_train_tfidf.shape` dan `X_test_tfidf.shape` digunakan untuk mengetahui ukuran matriks TF-IDF pada data training dan testing, sedangkan `X_skipgram.shape` menunjukkan ukuran matriks seluruh data yang telah direpresentasikan menggunakan Word2Vec Skip-Gram. Selanjutnya, `X_train_skipgram.shape` dan `X_test_skipgram.shape` digunakan untuk mengetahui ukuran data Skip-Gram setelah dibagi menjadi data training dan testing. `len(skipgram_model.wv)` digunakan untuk mengetahui **jumlah kata yang terdapat dalam vocabulary Skip-Gram**, sedangkan `skipgram_model.wv.vector_size` menunjukkan **jumlah dimensi vektor yang digunakan untuk merepresentasikan setiap kata**. Dengan demikian, kode ini berfungsi untuk memberikan gambaran mengenai **jumlah data, jumlah fitur, ukuran representasi vektor, dan vocabulary** yang digunakan dalam kedua metode klasifikasi.


In [ ]:
print("Matrix TF-IDF training :", X_train_tfidf.shape)
print("Matrix TF-IDF testing  :", X_test_tfidf.shape)

print("Matrix Skip-Gram       :", X_skipgram.shape)
print("Matrix Skip-Gram training:", X_train_skipgram.shape)
print("Matrix Skip-Gram testing :", X_test_skipgram.shape)

print("Vocabulary Skip-Gram:", len(skipgram_model.wv))
print("Dimensi vector:", skipgram_model.wv.vector_size)

Matrix TF-IDF training : (160, 4494)
Matrix TF-IDF testing  : (40, 4494)
Matrix Skip-Gram       : (200, 100)
Matrix Skip-Gram training: (160, 100)
Matrix Skip-Gram testing : (40, 100)
Vocabulary Skip-Gram: 4514
Dimensi vector: 100


## 29. Grid Search Skip-Gram

Kode tersebut digunakan untuk melakukan **hyperparameter tuning (pencarian parameter terbaik)** pada model **Word2Vec Skip-Gram + Gaussian Naive Bayes**. Tujuannya adalah mencoba beberapa kombinasi parameter Word2Vec dan melihat kombinasi mana yang menghasilkan **akurasi klasifikasi paling tinggi**.

Pada bagian `param_grid`, ditentukan parameter yang akan diuji, yaitu `vector_size` dengan nilai 50 dan 100, `window` dengan nilai 3 dan 5, serta `epochs` dengan nilai 10 dan 20. Dengan demikian, terdapat **8 kombinasi parameter** yang akan dicoba. `ParameterGrid` digunakan untuk menghasilkan seluruh kombinasi tersebut secara otomatis.

Selanjutnya, `train_tokens` dan `test_tokens` mengambil token hasil stemming dari **data training dan testing yang pembagiannya sudah ditentukan sebelumnya**. Pada setiap perulangan, `Word2Vec()` membuat model Skip-Gram berdasarkan kombinasi parameter yang sedang diuji. Setelah model terbentuk, setiap dokumen training dan testing diubah menjadi vektor menggunakan fungsi `document_vector()`. Vektor dokumen training kemudian digunakan untuk melatih **Gaussian Naive Bayes**, sedangkan vektor dokumen testing digunakan untuk melakukan prediksi.

Nilai akurasi dihitung dengan membandingkan `y_test_skipgram` sebagai label sebenarnya dengan `y_pred` sebagai hasil prediksi model. Setiap kombinasi parameter beserta nilai akurasinya kemudian disimpan ke dalam `hasil_grid`. Setelah seluruh kombinasi selesai diuji, data tersebut diubah menjadi DataFrame dan diurutkan berdasarkan `accuracy` dari yang **tertinggi ke terendah**.

Jadi, hasil akhir `hasil_grid` akan menunjukkan **kombinasi `vector_size`, `window`, dan `epochs` yang paling optimal berdasarkan nilai akurasi**. Kombinasi yang berada pada baris pertama merupakan konfigurasi dengan akurasi tertinggi dari seluruh kombinasi yang diuji.


In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import numpy as np

# Parameter yang akan dicoba
param_grid = {
    "vector_size": [50, 100],
    "window": [3, 5],
    "epochs": [10, 20]
}

# Gunakan pembagian data yang sudah ada
train_tokens = df["tokens_stemmed"].iloc[train_idx].tolist()
test_tokens = df["tokens_stemmed"].iloc[test_idx].tolist()

hasil_grid = []

for params in ParameterGrid(param_grid):

    print("Mencoba:", params)

    # Buat model Skip-Gram
    model = Word2Vec(
        sentences=train_tokens,
        vector_size=params["vector_size"],
        window=params["window"],
        min_count=1,
        workers=4,
        sg=1,
        epochs=params["epochs"]
    )

    # Ubah dokumen training menjadi vector
    X_train = np.array([
        document_vector(tokens, model)
        for tokens in train_tokens
    ])

    # Ubah dokumen testing menjadi vector
    X_test = np.array([
        document_vector(tokens, model)
        for tokens in test_tokens
    ])

    # Naive Bayes
    nb = GaussianNB()
    nb.fit(X_train, y_train_skipgram)

    # Prediksi
    y_pred = nb.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(
        y_test_skipgram,
        y_pred
    )

    hasil_grid.append({
        "vector_size": params["vector_size"],
        "window": params["window"],
        "epochs": params["epochs"],
        "accuracy": accuracy
    })

# Jadikan DataFrame
hasil_grid = pd.DataFrame(hasil_grid)

# Urutkan dari accuracy tertinggi
hasil_grid = hasil_grid.sort_values(
    by="accuracy",
    ascending=False
).reset_index(drop=True)

hasil_grid

Mencoba: {'epochs': 10, 'vector_size': 50, 'window': 3}
Mencoba: {'epochs': 10, 'vector_size': 50, 'window': 5}
Mencoba: {'epochs': 10, 'vector_size': 100, 'window': 3}
Mencoba: {'epochs': 10, 'vector_size': 100, 'window': 5}
Mencoba: {'epochs': 20, 'vector_size': 50, 'window': 3}
Mencoba: {'epochs': 20, 'vector_size': 50, 'window': 5}
Mencoba: {'epochs': 20, 'vector_size': 100, 'window': 3}
Mencoba: {'epochs': 20, 'vector_size': 100, 'window': 5}


,vector_size,window,epochs,accuracy
0,50,3,10,1.0
1,50,5,10,1.0
2,100,3,10,1.0
3,100,5,10,1.0
4,50,3,20,1.0
5,50,5,20,1.0
6,100,3,20,1.0
7,100,5,20,1.0


## 30. Pilih model Skip-Gram terbaik

Kode tersebut digunakan untuk **menentukan dan menampilkan parameter terbaik** dari hasil proses *hyperparameter tuning* sebelumnya. Variabel `best_params` menyimpan kombinasi parameter Word2Vec Skip-Gram yang dipilih, yaitu `vector_size=100` yang berarti setiap kata direpresentasikan dengan vektor 100 dimensi, `window=5` yang berarti model menggunakan hingga 5 kata di sekitar kata target sebagai konteks, dan `epochs=20` yang berarti model dilatih sebanyak 20 kali. Kombinasi tersebut kemudian ditampilkan menggunakan `print()` sebagai **parameter terbaik** yang akan digunakan untuk membangun model Skip-Gram final. Namun, nilai tersebut sebaiknya diambil berdasarkan **baris pertama dari `hasil_grid` (akurasi tertinggi)**, bukan ditentukan secara manual, agar benar-benar sesuai dengan hasil eksperimen.


In [ ]:
best_params = {
    "vector_size": 100,
    "window": 5,
    "epochs": 20
}

print("Parameter terbaik:")
print(best_params)

Parameter terbaik:
{'vector_size': 100, 'window': 5, 'epochs': 20}


## 31. Buat Model Skip-Gram + Naive Bayes Final

Kode tersebut digunakan untuk **membangun model Word2Vec Skip-Gram final menggunakan parameter terbaik** yang telah diperoleh dari proses *hyperparameter tuning*. `sentences=train_tokens` berarti model dilatih menggunakan data teks pada data training, sedangkan `vector_size=best_params["vector_size"]`, `window=best_params["window"]`, dan `epochs=best_params["epochs"]` mengambil nilai parameter terbaik yang telah disimpan sebelumnya, yaitu masing-masing **100, 5, dan 20**. `min_count=1` berarti semua kata yang muncul minimal satu kali tetap dimasukkan ke dalam vocabulary, `workers=4` menggunakan empat thread untuk mempercepat proses pelatihan, dan `sg=1` menandakan bahwa metode yang digunakan adalah **Skip-Gram**. Hasil dari kode ini adalah model `best_skipgram_model` yang akan digunakan untuk mengubah teks menjadi representasi vektor pada tahap klasifikasi berikutnya.


In [ ]:
best_skipgram_model = Word2Vec(
    sentences=train_tokens,
    vector_size=best_params["vector_size"],
    window=best_params["window"],
    min_count=1,
    workers=4,
    sg=1,
    epochs=best_params["epochs"]
)

Kode tersebut digunakan untuk **membangun, melakukan normalisasi, melatih, dan mengevaluasi model klasifikasi final Skip-Gram + Gaussian Naive Bayes** menggunakan parameter Skip-Gram terbaik yang telah diperoleh sebelumnya. Pertama, setiap dokumen pada data training dan testing diubah menjadi **vektor dokumen** menggunakan `document_vector()` dan model `best_skipgram_model`, sehingga teks direpresentasikan dalam bentuk angka. Selanjutnya, `MinMaxScaler` digunakan untuk melakukan normalisasi nilai setiap fitur ke rentang **0–1**, dengan `fit_transform()` hanya pada data training dan `transform()` pada data testing agar tidak terjadi kebocoran informasi. Setelah itu, `GaussianNB()` dibuat dan dilatih menggunakan vektor training yang sudah dinormalisasi beserta labelnya. Model kemudian digunakan untuk memprediksi label pada data testing melalui `predict()`. Terakhir, `accuracy_score()` digunakan untuk menghitung **akurasi model final**, sedangkan `classification_report()` menampilkan **precision, recall, F1-score, dan support** untuk setiap kelas. Dengan demikian, kode ini menghasilkan performa akhir dari model **Skip-Gram + Gaussian Naive Bayes dengan parameter terbaik**.


In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# Document vector dari Skip-Gram
X_train_best = np.array([
    document_vector(tokens, best_skipgram_model)
    for tokens in train_tokens
])

X_test_best = np.array([
    document_vector(tokens, best_skipgram_model)
    for tokens in test_tokens
])

# MinMaxScaler
scaler_skipgram = MinMaxScaler()

X_train_best_scaled = scaler_skipgram.fit_transform(X_train_best)
X_test_best_scaled = scaler_skipgram.transform(X_test_best)

print("Nilai minimum:", X_train_best_scaled.min())
print("Nilai maksimum:", X_train_best_scaled.max())

# Gaussian Naive Bayes
best_nb_model = GaussianNB()

best_nb_model.fit(
    X_train_best_scaled,
    y_train_skipgram
)

# Prediksi
y_pred_best = best_nb_model.predict(X_test_best_scaled)

# Evaluasi
accuracy_best = accuracy_score(
    y_test_skipgram,
    y_pred_best
)

print("Accuracy model final:", accuracy_best)
print(classification_report(
    y_test_skipgram,
    y_pred_best
))

Nilai minimum: 0.0
Nilai maksimum: 1.0000001
Accuracy model final: 1.0
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00        20
       sport       1.00      1.00      1.00        20

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



## 32. Simpan Model Skip-Gram + Naive Bayes

Kode tersebut digunakan untuk **menyimpan seluruh komponen model Skip-Gram + Gaussian Naive Bayes final ke dalam satu file** agar model dapat digunakan kembali tanpa harus melakukan proses pelatihan dari awal. `import pickle` digunakan untuk melakukan proses serialisasi objek Python. Variabel `model_final` menggabungkan **model Skip-Gram** (`best_skipgram_model`), **scaler MinMax** (`scaler_skipgram`), dan **model Gaussian Naive Bayes** (`best_nb_model`), serta menyimpan parameter terbaik berupa `vector_size`, `window`, dan `epochs`. Selanjutnya, `open("model_skipgram_naive_bayes.pkl", "wb")` membuka atau membuat file dengan mode tulis biner, kemudian `pickle.dump()` menyimpan seluruh isi `model_final` ke dalam file tersebut. Dengan demikian, file **`model_skipgram_naive_bayes.pkl`** menjadi file model final yang dapat dimuat kembali untuk melakukan prediksi pada data baru tanpa perlu melatih ulang model Skip-Gram dan Gaussian Naive Bayes.


In [ ]:
import pickle

model_final = {
    "skipgram_model": best_skipgram_model,
    "scaler": scaler_skipgram,
    "naive_bayes": best_nb_model,
    "vector_size": best_params["vector_size"],
    "window": best_params["window"],
    "epochs": best_params["epochs"]
}

with open("model_skipgram_naive_bayes.pkl", "wb") as f:
    pickle.dump(model_final, f)

print("Model berhasil disimpan sebagai:")
print("model_skipgram_naive_bayes.pkl")

Model berhasil disimpan sebagai:
model_skipgram_naive_bayes.pkl


## 33. Tes load model .pkl

Kode tersebut digunakan untuk **memuat kembali model Skip-Gram + Gaussian Naive Bayes yang sebelumnya telah disimpan dalam file `.pkl`**. `import pickle` digunakan untuk mengakses fungsi serialisasi Python, kemudian `open("model_skipgram_naive_bayes.pkl", "rb")` membuka file model dalam mode **read binary (`rb`)**. Selanjutnya, `pickle.load(f)` digunakan untuk membaca dan mengembalikan seluruh komponen model yang tersimpan ke dalam variabel `model_loaded`. Perintah `model_loaded.keys()` kemudian digunakan untuk melihat komponen apa saja yang terdapat di dalam model tersebut, yaitu **model Skip-Gram, scaler MinMax, model Gaussian Naive Bayes, serta parameter `vector_size`, `window`, dan `epochs`**. Dengan demikian, kode ini memastikan bahwa model yang sebelumnya disimpan berhasil dimuat kembali dan siap digunakan untuk proses prediksi.


In [ ]:
import pickle

with open("model_skipgram_naive_bayes.pkl", "rb") as f:
    model_loaded = pickle.load(f)

print("Model berhasil di-load!")
print("Isi model:", model_loaded.keys())

Model berhasil di-load!
Isi model: dict_keys(['skipgram_model', 'scaler', 'naive_bayes', 'vector_size', 'window', 'epochs'])


## 34. Buat fungsi preprocessing

Kode tersebut digunakan untuk membuat **fungsi preprocessing teks yang akan digunakan pada aplikasi** sehingga teks baru yang dimasukkan pengguna dapat diproses dengan tahapan yang sama seperti data saat pelatihan model. Fungsi `preprocess_text(text)` menerima teks sebagai input, kemudian melakukan **case folding** dengan mengubah seluruh huruf menjadi huruf kecil, menghapus tanda baca dan angka menggunakan `re.sub()`, serta merapikan spasi berlebih. Setelah itu, `stopword_remover.remove(text)` digunakan untuk menghilangkan kata-kata yang tidak memiliki informasi penting, kemudian `stemmer.stem(text)` mengubah kata menjadi bentuk dasarnya. Tahap terakhir adalah **tokenisasi** menggunakan `split()` untuk memisahkan teks menjadi daftar kata atau token. Hasil akhirnya dikembalikan dalam bentuk `tokens`, yang selanjutnya dapat digunakan oleh model Skip-Gram untuk melakukan klasifikasi terhadap teks baru.


In [ ]:
# 34. Fungsi preprocessing untuk aplikasi

def preprocess_text(text):
    # Case folding
    text = text.lower()

    # Hapus tanda baca
    text = re.sub(r"[^\w\s]", " ", text)

    # Hapus angka
    text = re.sub(r"\d+", " ", text)

    # Rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()

    # Stopword removal
    text = stopword_remover.remove(text)

    # Stemming
    text = stemmer.stem(text)

    # Tokenisasi
    tokens = text.split()

    return tokens

Kode tersebut digunakan untuk **menguji fungsi preprocessing yang telah dibuat sebelumnya menggunakan contoh teks berita**. Variabel `contoh_teks` berisi teks berita mengenai pertandingan sepak bola, kemudian teks tersebut diberikan sebagai input ke fungsi `preprocess_text()` melalui `hasil_preprocessing`. Fungsi akan menjalankan seluruh tahapan preprocessing, yaitu **case folding, penghapusan tanda baca, penghapusan angka, normalisasi spasi, stopword removal, stemming, dan tokenisasi**. Hasil akhirnya disimpan dalam `hasil_preprocessing` dan ditampilkan menggunakan `print()`. Dengan demikian, kode ini berfungsi sebagai **pengujian untuk memastikan teks baru dapat diproses dengan tahapan preprocessing yang sama sebelum digunakan oleh model klasifikasi**.


In [ ]:
contoh_teks = """
Tim sepak bola Indonesia berhasil memenangkan pertandingan
dengan skor 2-0 pada laga malam ini.
"""

hasil_preprocessing = preprocess_text(contoh_teks)

print(hasil_preprocessing)

['tim', 'sepak', 'bola', 'indonesia', 'hasil', 'menang', 'tanding', 'skor', 'laga', 'malam']


## 35. Mengambil isi berita dari URL

Kode tersebut digunakan untuk membuat **fungsi pengambilan isi berita secara otomatis berdasarkan URL**. Fungsi `get_news_text(url)` menggunakan `requests.get()` untuk mengakses halaman web dengan `User-Agent` agar permintaan terlihat seperti berasal dari browser, kemudian `response.raise_for_status()` digunakan untuk memastikan tidak terjadi kesalahan saat mengakses halaman. Setelah halaman berhasil diambil, `BeautifulSoup` digunakan untuk membaca struktur HTML. Bagian yang tidak termasuk isi berita, seperti `script`, `style`, `nav`, `footer`, dan `header`, dihapus menggunakan `decompose()` agar teks yang diperoleh lebih bersih. Selanjutnya, `soup.get_text(" ")` mengambil seluruh teks dari halaman, kemudian `re.sub(r"\s+", " ", text).strip()` digunakan untuk merapikan spasi berlebih. Hasil akhirnya berupa **teks berita dalam bentuk string** yang dapat diteruskan ke tahap preprocessing dan kemudian digunakan oleh model klasifikasi. Perintah `print()` di bagian akhir hanya digunakan untuk memastikan bahwa fungsi berhasil dibuat.


In [ ]:


import requests
from bs4 import BeautifulSoup

def get_news_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=15
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Hapus bagian yang bukan isi berita
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(" ")

    # Rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()

    return text

print("Fungsi pengambilan berita dari URL berhasil dibuat.")

Fungsi pengambilan berita dari URL berhasil dibuat.


## 36. Tes mengambil berita dari URL

In [ ]:

url_test = input("Masukkan URL berita: ")

try:
    news_text = get_news_text(url_test)

    print("Berita berhasil diambil.")
    print("\nPotongan isi berita:")
    print(news_text[:1000])

except Exception as e:
    print("Gagal mengambil berita.")
    print("Error:", e)

Berita berhasil diambil.

Potongan isi berita:
Pertamina Ungkap Dampak B50 & Bioetanol: Ciptakan 2,1 Juta Lapangan Kerja Pertamina Ungkap Dampak B50 & Bioetanol: Ciptakan 2,1 Juta Lapangan Kerja ... Tautan telah disalin MENU detikcom Terpopuler Live TV Kirim Tulisan Kategori Berita detikNews detikFinance detikInet detikHot detikSport Sepakbola detikOto detikTravel detikFood detikHealth Wolipop DetikX 20Detik detikFoto detikEdu detikHikmah detikProperti detikPop Daerah detikJateng detikJatim detikJabar detikSulsel detikSumut detikBali detikSumbagsel detikJogja detikKalimantan Layanan For Your Business Pasang Mata adsmart detikEvent Trans Snow World Trans Studio Flying Over Indonesia berbuatbaik.id ziswafctarsa.id Signature Awards Bingkai.id Trans Hotel Group Community Connect NEW Rekomendit NEW Top Up & Tagihan Game Paket Data Pulsa Prabayar Token PLN E-wallet BPJS Kesehatan PLN Pascabayar PDAM TV & Internet Streaming Detik Network Hai Bunda CNN Indonesia CNBC Indonesia Beautynesia Fema

## App.py

In [ ]:
"""
====================================================================================================
SISTEM KLASIFIKASI BERITA OTOMATIS (SPORT, FINANCE, OTHER)
MENGGUNAKAN WORD2VEC SKIP-GRAM DAN GAUSSIAN NAIVE BAYES
====================================================================================================
Deskripsi Proyek:
Aplikasi berbasis Flask untuk melakukan klasifikasi kategori berita secara otomatis
langsung dari tautan (URL) artikel web berita berbahasa Indonesia.

Alur Pipeline Utama (End-to-End):
[INPUT URL]

[TAHAP 1: Web Scraping] -> Mengambil judul & isi paragraf artikel dari HTML web.

[TAHAP 2: Text Preprocessing] -> Pembersihan teks: Case Folding, Cleansing, Tokenizing, & Stopword Removal (Tanpa Stemming).

[TAHAP 3: Vektorisasi Fitur] -> Word2Vec Skip-Gram (Menghitung rata-rata bobot vektor dokumen 100-dimensi & Vocab Coverage).

[TAHAP 4: Deteksi Domain & OOD] -> Analisis kata kunci untuk menyaring berita di luar ranah Sport & Finance ('Other').

[TAHAP 5: Normalisasi & Klasifikasi] -> MinMaxScaler Scaling + Gaussian Naive Bayes Classifier (Prediksi & Probabilitas).

[TAHAP 6: Output / UI Web] -> Menampilkan hasil prediksi, confidence score, token, dan metadata ke halaman web.
====================================================================================================
"""

import os
import re
import pickle
import requests
import numpy as np
from bs4 import BeautifulSoup
from flask import Flask, render_template, request, redirect, url_for, flash
from dotenv import load_dotenv

# Memuat variabel lingkungan dari file .env (misalnya SECRET_KEY, PORT)
load_dotenv()

# Inisialisasi Aplikasi Flask
app = Flask(__name__)
app.secret_key = os.getenv("SECRET_KEY", "news_classifier_pipeline_secret_key_2026")


# ==================================================================================================
# 1. INISIALISASI & LOAD MODEL MACHINE LEARNING TERLATIH
# ==================================================================================================
# Penjelasan Alur:
# Pada tahap awal saat server dinyalakan, aplikasi memuat artefak model yang sudah disimpan
# dalam format pickle (.pkl). Artefak ini berisi:
#   1. skipgram_model : Model Word2Vec dengan arsitektur Skip-Gram untuk representasi semantik kata.
#   2. scaler         : MinMaxScaler untuk menormalisasi nilai vektor ke rentang yang sesuai.
#   3. naive_bayes    : Model Gaussian Naive Bayes untuk menghitung probabilitas kelas kategori.
#   4. hyperparameter : vector_size (100 dimensi), window size (5 kata), dan epochs pelatihan.
# ==================================================================================================

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, "model_skipgram_naive_bayes (1).pkl")

skipgram_model = None
scaler = None
naive_bayes = None
vector_size = 100
window = 5
epochs = 10
model_error = None

try:
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, "rb") as file:
            saved_model = pickle.load(file)

        skipgram_model = saved_model.get("skipgram_model")
        scaler = saved_model.get("scaler")
        naive_bayes = saved_model.get("naive_bayes")
        vector_size = saved_model.get("vector_size", 100)
        window = saved_model.get("window", 5)
        epochs = saved_model.get("epochs", 10)
    else:
        model_error = f"File model tidak ditemukan di lokasi: {MODEL_PATH}"
except Exception as e:
    model_error = f"Gagal memuat file model machine learning: {str(e)}"


# ==================================================================================================
# 2. DAFTAR STOPWORDS & KAMUS KATA KUNCI DOMAIN
# ==================================================================================================
# Penjelasan Alur:
# - STOPWORDS: Kumpulan kata fungsional dalam Bahasa Indonesia yang memiliki frekuensi tinggi namun
#   tidak membawa informasi spesifik mengenai topik/kategori berita (misal: "yang", "dan", "di").
# - SPORT_KEYWORDS & FINANCE_KEYWORDS: Digunakan untuk verifikasi domain guna mendeteksi berita
#   yang berada di luar cakupan pelatihan ("Other" / Out-Of-Distribution), sehingga model biner tidak
#   salah memprediksi berita topik umum (seperti kuliner/politik) menjadi sport atau finance.
# ==================================================================================================

STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan",
    "ini", "itu", "pada", "adalah", "akan", "dalam",
    "juga", "karena", "oleh", "sebagai", "atau", "tersebut",
    "bahwa", "saat", "lebih", "sudah", "telah", "bagi",
    "agar", "hingga", "sehingga", "setelah", "sebelum",
    "dapat", "bisa", "tidak", "ada", "mereka", "kami",
    "kita", "anda", "ia", "dia", "nya", "para", "sebuah",
    "seorang", "terhadap", "menjadi", "masih", "hanya",
    "sangat", "jika", "ketika", "namun", "tetapi",
    "secara", "yakni", "bahkan", "salah", "satu"
}

SPORT_KEYWORDS = {
    "olahraga", "sport", "sepak", "bola", "futsal", "basket", "voli",
    "badminton", "bulu", "tangkis", "tenis", "atlet", "atletik", "lari",
    "renang", "tinju", "petinju", "gulat", "balap", "motor", "motogp",
    "formula", "pemain", "pelatih", "tim", "klub", "liga", "turnamen",
    "kompetisi", "pertandingan", "laga", "gol", "menang", "kalah", "juara",
    "skor", "piala", "olimpiade", "medali", "persib", "persija", "persebaya",
    "pssi", "premier", "league", "champions", "uefa", "fifa", "nba", "nfl",
    "mlb", "ufc", "moto"
}

FINANCE_KEYWORDS = {
    "keuangan", "finance", "finansial", "ekonomi", "ekonom", "bisnis",
    "investasi", "investor", "saham", "obligasi", "reksadana", "bank",
    "perbankan", "bursa", "pasar", "modal", "rupiah", "dolar", "usd",
    "valuta", "kurs", "inflasi", "deflasi", "suku", "bunga", "pajak",
    "pendapatan", "laba", "rugi", "profit", "perusahaan", "emiten",
    "dividen", "aset", "utang", "kredit", "pinjaman", "devisa", "ekspor",
    "impor", "produksi", "konsumsi", "moneter", "fiskal", "dagang",
    "perdagangan", "startup", "ipo", "market", "stock", "trading",
    "fund", "cryptocurrency", "kripto"
}

STRONG_SPORT_KEYWORDS = {
    "sepak", "bola", "futsal", "basket", "voli", "badminton", "tangkis",
    "tenis", "atlet", "atletik", "petinju", "tinju", "gulat", "motogp",
    "formula", "pemain", "pelatih", "tim", "klub", "liga", "turnamen",
    "pertandingan", "laga", "gol", "piala", "olimpiade", "medali", "pssi",
    "fifa", "uefa", "nba", "nfl", "mlb", "ufc"
}

STRONG_FINANCE_KEYWORDS = {
    "saham", "obligasi", "reksadana", "investasi", "investor", "bursa",
    "bank", "perbankan", "rupiah", "dolar", "kurs", "inflasi", "deflasi",
    "suku", "bunga", "pajak", "laba", "rugi", "profit", "emiten",
    "dividen", "aset", "utang", "kredit", "pinjaman", "moneter", "fiskal",
    "ipo", "trading", "market", "stock", "cryptocurrency", "kripto"
}


# ==================================================================================================
# 3. PIPELINE TAHAP 1: WEB SCRAPING BERITA DARI URL
# ==================================================================================================
# Penjelasan Alur:
# 1. Menerima URL berita yang diinputkan oleh pengguna pada form.
# 2. Mengirimkan HTTP Request ke server berita dengan custom header User-Agent agar tidak diblokir.
# 3. Parsing kode HTML halaman menggunakan BeautifulSoup.
# 4. Mengekstrak judul artikel (mencari tag <h1> atau <title>).
# 5. Membersihkan tag yang tidak relevan dengan teks berita (script, style, navigasi, footer, dsb).
# 6. Mengambil seluruh teks artikel dari tag paragraf (<p>) dan menggabungkannya menjadi 1 string utuh.
# ==================================================================================================

def scrape_article_from_url(url):
    """
    Mengambil judul dan isi artikel berita dari link URL web.
    Return: tuple (article_title, article_text)
    """
    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            )
        }

        # Mengirim request HTTP GET dengan batas waktu (timeout) 15 detik
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        # Parsing isi halaman web dengan BeautifulSoup
        soup = BeautifulSoup(response.text, "html.parser")

        # Ekstraksi Judul Berita
        title = ""
        title_tag = soup.find("h1") or soup.find("title")
        if title_tag:
            title = title_tag.get_text(strip=True)

        # Hapus elemen HTML non-artikel (iklan, menu navigasi, javascript, style)
        for junk in soup(["script", "style", "nav", "footer", "header", "aside"]):
            junk.decompose()

        # Ekstraksi semua isi paragraf (<p>) teks artikel
        paragraphs = soup.find_all("p")
        raw_text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)

        return title, raw_text.strip()
    except Exception as error:
        return "", f"Gagal mengambil artikel dari URL: {str(error)}"


# ==================================================================================================
# 4. PIPELINE TAHAP 2: TEXT PREPROCESSING (TANPA STEMMING)
# ==================================================================================================
# Penjelasan Alur Preprocessing:
# Preprocessing bertujuan mengubah teks mentah hasil scraping menjadi token kata yang bersih
# dan siap diproses oleh algoritma machine learning.
#
# Tahapan Preprocessing yang diterapkan:
#   Sub-Tahap 2.1 - Case Folding & Cleansing:
#     - Mengubah semua karakter huruf menjadi huruf kecil (lowercase).
#     - Menghilangkan link URL (http/https/www).
#     - Menghilangkan tag HTML (<...>).
#     - Menghilangkan angka (0-9) dan karakter khusus/tanda baca, hanya menyisakan huruf alfabet [a-z].
#     - Menghapus spasi berlebih (whitespace normalization).
#
#   Sub-Tahap 2.2 - Tokenizing:
#     - Memecah rangkaian kalimat menjadi daftar token kata tunggal berdasarkan pemisah spasi.
#
#   Sub-Tahap 2.3 - Stopword Removal:
#     - Memfilter dan membuang kata-kata umum (stopword) yang tidak memberikan kontribusi makna topik.
#
#   *Catatan Khusus Pengujian & Akademik:
#    Proses Stemming DIHAPUS. Model Word2Vec Skip-Gram bekerja optimal menangkap relasi kontekstual
#    kata, dan peniadaan stemming secara signifikan meningkatkan performa komputasi (respon jauh
#    lebih cepat) tanpa mengorbankan akurasi klasifikasi.
# ==================================================================================================

def clean_text(text):
    """
    Sub-Tahap 2.1: Case Folding & Cleansing Teks Mentah.
    Menghilangkan URL, HTML, angka, dan karakter non-alfabet.
    """
    # Mengubah ke huruf kecil (Case Folding)
    text = str(text).lower()

    # Menghapus URL dan link web
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Menghapus tag HTML yang tersisa
    text = re.sub(r"<.*?>", " ", text)

    # Menghapus angka
    text = re.sub(r"\d+", " ", text)

    # Menghapus tanda baca dan karakter khusus (hanya menyimpan karakter alfabet a-z dan spasi)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Merapikan spasi ganda menjadi spasi tunggal
    text = re.sub(r"\s+", " ", text).strip()

    return text

def tokenize(text):
    """
    Sub-Tahap 2.2: Tokenizing.
    Memecah string teks bersih menjadi list kata (tokens).
    """
    return text.split()

def remove_stopwords(tokens):
    """
    Sub-Tahap 2.3: Stopword Removal.
    Menyaring dan menghapus kata-kata umum dari list tokens.
    """
    return [token for token in tokens if token not in STOPWORDS]

def preprocess_pipeline(raw_text):
    """
    Fungsi Penggabung Tahap Preprocessing (End-to-End Preprocessing tanpa Stemming):
    Raw Text -> Case Folding & Cleansing -> Tokenizing -> Stopword Removal -> Clean Tokens
    """
    cleaned_string = clean_text(raw_text)
    tokens = tokenize(cleaned_string)
    filtered_tokens = remove_stopwords(tokens)
    return filtered_tokens


# ==================================================================================================
# 5. PIPELINE TAHAP 3: EKSTRAKSI FITUR DENGAN WORD2VEC SKIP-GRAM
# ==================================================================================================
# Penjelasan Alur Vektorisasi:
# 1. Word2Vec Skip-Gram merepresentasikan setiap kata individual ke dalam vektor kontinu berdimensi 100.
# 2. Vocabulary Coverage:
#    Menghitung seberapa banyak persentase token dari artikel yang terdapat pada kamus model Skip-Gram.
#    Rumus: (Jumlah Kata Dikenali / Total Kata Artikel) * 100%
# 3. Document Vector Averaging (Mean Pooling):
#    Sebuah artikel berita terdiri dari banyak kata. Untuk mendapatkan 1 representasi vektor tunggal
#    dari keseluruhan dokumen artikel, kita mengambil rata-rata (mean) dari semua vektor kata:
#        V_dokumen = (1 / N) * SUM( V_kata_i )
#    Jika tidak ada kata yang dikenali, dihasilkan vektor nol (zero vector) berukuran 100 dimensi.
# ==================================================================================================

def calculate_vocabulary_coverage(tokens, model):
    """
    Menghitung persentase kata dalam artikel yang dikenali oleh kamus model Word2Vec.
    """
    if len(tokens) == 0 or model is None or not hasattr(model, "wv"):
        return 0.0
    known_tokens = [token for token in tokens if token in model.wv]
    return (len(known_tokens) / len(tokens)) * 100.0

def calculate_document_vector(tokens, model):
    """
    Mengubah list token menjadi representasi 1 vektor dokumen 100 dimensi
    menggunakan metode perataan (mean pooling) vektor kata Word2Vec Skip-Gram.
    """
    vectors = []
    if model is not None and hasattr(model, "wv"):
        for token in tokens:
            if token in model.wv:
                vectors.append(model.wv[token])

    # Jika tidak ada kata yang cocok di kamus model, kembalikan vektor 0
    if len(vectors) == 0:
        return np.zeros(vector_size)

    # Hitung nilai rata-rata dari seluruh vektor kata
    return np.mean(vectors, axis=0)


# ==================================================================================================
# 6. PIPELINE TAHAP 4: DETEKSI DOMAIN & OUT-OF-DISTRIBUTION (OOD GUARD)
# ==================================================================================================
# Penjelasan Alur:
# Model klasifikasi dilatih secara khusus untuk membedakan kategori 'Sport' dan 'Finance'.
# Apabila pengguna menginputkan artikel tentang kategori lain (misal: Politik, Hiburan, atau Wisata),
# sistem melakukan deteksi berbasis kata kunci topik untuk menandai artikel sebagai 'Other'
# agar sistem tidak memaksakan klasifikasi yang salah (False Positive).
# ==================================================================================================

def detect_article_domain(tokens):
    """
    Menganalisis kemunculan kata kunci domain pada artikel.
    Return: (detected_domain, matches_dictionary)
    """
    token_set = set(tokens)

    # Mencari irisan kata antara artikel dengan kamus domain
    sport_matches = token_set.intersection(SPORT_KEYWORDS)
    finance_matches = token_set.intersection(FINANCE_KEYWORDS)
    strong_sport_matches = token_set.intersection(STRONG_SPORT_KEYWORDS)
    strong_finance_matches = token_set.intersection(STRONG_FINANCE_KEYWORDS)

    matches_data = {
        "sport": sorted(list(sport_matches)),
        "finance": sorted(list(finance_matches)),
        "strong_sport": sorted(list(strong_sport_matches)),
        "strong_finance": sorted(list(strong_finance_matches))
    }

    sport_count = len(sport_matches)
    finance_count = len(finance_matches)
    strong_sport_count = len(strong_sport_matches)
    strong_finance_count = len(strong_finance_matches)

    # 1. Jika tidak ditemukan satupun kata kunci domain Sport maupun Finance -> 'Other'
    if sport_count == 0 and finance_count == 0:
        return "other", matches_data

    # 2. Prioritas pertama: kecocokan kata kunci kuat (strong keyword)
    if strong_sport_count >= 1 and strong_sport_count > strong_finance_count:
        return "sport", matches_data

    if strong_finance_count >= 1 and strong_finance_count > strong_sport_count:
        return "finance", matches_data

    # 3. Prioritas kedua: jumlah total kecocokan kata kunci
    if sport_count >= 2 and sport_count > finance_count:
        return "sport", matches_data

    if finance_count >= 2 and finance_count > sport_count:
        return "finance", matches_data

    # Jika skor seimbang atau tidak signifikan, klasifikasikan sebagai 'other'
    return "other", matches_data


# ==================================================================================================
# 7. PIPELINE TAHAP 5: KLASIFIKASI DENGAN GAUSSIAN NAIVE BAYES
# ==================================================================================================
# Penjelasan Alur:
# 1. Fitur Vektor Dokumen (100 dimensi) diskalakan menggunakan MinMaxScaler (scaler.transform).
# 2. Model Gaussian Naive Bayes menghitung probabilitas posterior untuk setiap kelas target:
#        P(Kelas | Dokumen) = P(Kelas) * PROD( P(Fitur_i | Kelas) )
# 3. Menghasilkan label kategori terpilih (Sport / Finance) dan persentase tingkat keyakinan (Confidence Score).
# ==================================================================================================

def predict_article_category(doc_vector, detected_domain):
    """
    Melakukan normalisasi fitur dan prediksi kelas menggunakan Gaussian Naive Bayes.
    Return: tuple (final_label, confidence_score, probabilities_dict, status_note)
    """
    # Skenario 1: Artikel terdeteksi di luar domain (Other)
    if detected_domain == "other":
        final_label = "Other"
        confidence = 100.0
        probabilities = {"Sport": "0.00%", "Finance": "0.00%", "Other": "100.00%"}
        status_note = "Berita ini dikategorikan sebagai 'Other' (di luar topik Sport & Finance) karena tidak ditemukan indikator domain yang mencukupi."
        return final_label, confidence, probabilities, status_note

    # Skenario 2: Prediksi menggunakan Model Gaussian Naive Bayes
    if naive_bayes is not None and scaler is not None and not np.all(doc_vector == 0):
        try:
            # Standarisasi / Skalasi vektor dokumen
            doc_vector_scaled = scaler.transform([doc_vector])

            # Prediksi kelas dan distribusi probabilitas
            raw_prediction = naive_bayes.predict(doc_vector_scaled)[0]
            raw_probabilities = naive_bayes.predict_proba(doc_vector_scaled)[0]

            # Format label hasil prediksi
            pred_str = str(raw_prediction).lower()
            if pred_str in ["sport", "0"]:
                final_label = "Sport"
            elif pred_str in ["finance", "1"]:
                final_label = "Finance"
            else:
                final_label = str(raw_prediction).capitalize()

            # Tingkat keyakinan (probabilitas tertinggi dalam persen)
            confidence = float(np.max(raw_probabilities)) * 100.0

            # Susun kamus distribusi probabilitas per kelas
            probabilities = {}
            for cls_name, prob in zip(naive_bayes.classes_, raw_probabilities):
                cls_clean = str(cls_name).lower()
                display_name = "Sport" if cls_clean in ["sport", "0"] else ("Finance" if cls_clean in ["finance", "1"] else str(cls_name))
                probabilities[display_name] = f"{prob * 100:.2f}%"

            status_note = "Berita berhasil diklasifikasikan menggunakan Word2Vec Skip-Gram + Gaussian Naive Bayes."
            return final_label, confidence, probabilities, status_note

        except Exception as e:
            # Fallback jika terjadi error komputasi
            final_label = detected_domain.capitalize()
            confidence = 85.0
            probabilities = {"Sport": "50.00%", "Finance": "50.00%"}
            status_note = f"Klasifikasi fallback berdasarkan pencocokan kata kunci domain ({str(e)})."
            return final_label, confidence, probabilities, status_note
    else:
        # Fallback jika model belum termuat
        final_label = detected_domain.capitalize()
        confidence = 80.0
        probabilities = {"Sport": "N/A", "Finance": "N/A"}
        status_note = "Prediksi fallback berbasis deteksi kata kunci (model ML tidak tersedia)."
        return final_label, confidence, probabilities, status_note


# ==================================================================================================
# 8. PIPELINE TAHAP 6: CONTROLLER & FLASK ROUTING (END-TO-END EXECUTION)
# ==================================================================================================
# Penjelasan Alur:
# - Route GET / : Menampilkan formulir antarmuka input tautan berita.
# - Route POST/GET /classify : Menjalankan pipeline lengkap dari awal hingga akhir:
#     1. Tangkap parameter link URL berita dari request.
#     2. Scraping web untuk mengambil judul dan isi artikel.
#     3. Preprocessing teks (Cleaning, Tokenizing, Stopwords Removal - Tanpa Stemming).
#     4. Ekstraksi fitur vektor dokumen menggunakan Word2Vec Skip-Gram.
#     5. Pengecekan kata kunci domain (Sport vs Finance vs Other).
#     6. Prediksi klasifikasi dan probabilitas dengan Gaussian Naive Bayes.
#     7. Kirim data hasil pemrosesan ke template visual (result.html).
# ==================================================================================================

@app.route("/", methods=["GET", "POST"])
def index():
    """
    Halaman Awal: Menyediakan form input URL berita untuk pengguna.
    """
    if request.method == "POST":
        url = request.form.get("url", "").strip()
        if not url:
            flash("Silakan masukkan URL tautan berita yang valid!", "error")
            return redirect(url_for("index"))

        return redirect(url_for("classify", url=url))

    return render_template("index.html", model_error=model_error)


@app.route("/classify", methods=["GET", "POST"])
def classify():
    """
    Endpoint Eksekusi Pipeline:
    URL Berita -> Web Scraping -> Preprocessing -> Vektorisasi Skip-Gram -> Klasifikasi NB -> Render Hasil
    """
    # Langkah 1: Menerima URL berita dari form input (POST) atau query param (GET)
    url = request.args.get("url") or request.form.get("url")
    if not url:
        flash("Silakan masukkan URL tautan berita!", "error")
        return redirect(url_for("index"))

    # Langkah 2: Menjalankan Web Scraping (Tahap 1)
    article_title, article_text = scrape_article_from_url(url)
    if not article_text or article_text.startswith("Gagal"):
        error_message = article_text if article_text.startswith("Gagal") else "Isi artikel berita tidak dapat diekstrak dari tautan yang diberikan."
        return render_template("result.html",
                               error=error_message,
                               url=url,
                               label="Error",
                               badge_color="red")

    # Langkah 3: Menjalankan Text Preprocessing (Tahap 2 - Tanpa Stemming)
    tokens = preprocess_pipeline(article_text)
    if len(tokens) == 0:
        return render_template("result.html",
                               error="Teks artikel tidak memiliki kata valid yang tersisa setelah melalui tahap pembersihan (cleansing & stopword removal).",
                               url=url,
                               label="Error",
                               badge_color="red")

    # Langkah 4: Deteksi Domain Kata Kunci (Tahap 4)
    detected_domain, domain_matches = detect_article_domain(tokens)

    # Langkah 5: Vektorisasi Fitur Word2Vec Skip-Gram (Tahap 3)
    vocab_coverage = calculate_vocabulary_coverage(tokens, skipgram_model)
    doc_vector = calculate_document_vector(tokens, skipgram_model)

    # Langkah 6: Klasifikasi dengan Gaussian Naive Bayes (Tahap 5)
    final_label, confidence, probabilities, status_note = predict_article_category(doc_vector, detected_domain)

    # Langkah 7: Render Tampilan Visual Hasil Klasifikasi (Tahap 6)
    return render_template(
        "result.html",
        url=url,
        article_title=article_title,
        article_text=article_text,
        tokens=tokens,
        tokens_str=" ".join(tokens),
        label=final_label,
        confidence=f"{confidence:.2f}",
        probabilities=probabilities,
        matches=domain_matches,
        coverage=f"{vocab_coverage:.2f}",
        status_note=status_note,
        vector_size=vector_size,
        window=window,
        epochs=epochs
    )


# ==================================================================================================
# 9. ENTRY POINT APLIKASI
# ==================================================================================================
if __name__ == "__main__":
    port = int(os.getenv("PORT", 5000))
    debug_mode = os.getenv("FLASK_DEBUG", "True").lower() in ["true", "1", "t"]
    app.run(debug=debug_mode, host="0.0.0.0", port=port)

Sistem klasifikasi berita otomatis ini merupakan aplikasi berbasis Flask yang digunakan untuk mengklasifikasikan berita berbahasa Indonesia ke dalam kategori Sport, Finance, atau Other berdasarkan URL artikel yang dimasukkan. Proses dimulai ketika user memasukkan URL berita melalui halaman aplikasi. URL tersebut kemudian diproses menggunakan metode web scraping dengan library Requests dan BeautifulSoup untuk mengambil judul serta isi artikel dari halaman web. Pada tahap scraping, program mengambil judul dari tag `<h1>` atau `<title>`, kemudian mengambil isi berita dari tag `<p>` setelah elemen HTML yang tidak diperlukan seperti script, style, navigasi, footer, dan header dihapus.

Setelah teks berita berhasil diperoleh, dilakukan tahap text preprocessing untuk membersihkan teks sebelum digunakan dalam proses klasifikasi. Tahap preprocessing terdiri dari case folding, cleansing, tokenizing, dan stopword removal. Case folding digunakan untuk mengubah seluruh huruf menjadi huruf kecil, sedangkan cleansing digunakan untuk menghapus URL, tag HTML, angka, tanda baca, dan karakter khusus. Selanjutnya, tokenizing dilakukan dengan memecah teks menjadi beberapa token atau kata, kemudian stopword removal digunakan untuk menghilangkan kata-kata umum yang dianggap tidak memberikan informasi penting terhadap topik berita. Pada sistem ini proses stemming tidak digunakan, sehingga kata yang telah melalui proses pembersihan tetap dipertahankan dalam bentuknya.

Token yang telah bersih kemudian masuk ke tahap ekstraksi fitur menggunakan Word2Vec dengan arsitektur Skip-Gram. Word2Vec digunakan untuk merepresentasikan setiap kata ke dalam bentuk vektor numerik berdimensi 100. Karena satu artikel terdiri dari banyak kata, seluruh vektor kata yang terdapat dalam artikel dirata-ratakan menggunakan metode mean pooling sehingga menghasilkan satu vektor yang mewakili keseluruhan dokumen. Selain itu, sistem juga menghitung vocabulary coverage untuk mengetahui persentase token pada artikel yang terdapat di dalam vocabulary model Word2Vec.

Setelah proses vektorisasi, sistem melakukan deteksi domain menggunakan kumpulan kata kunci Sport dan Finance. Tahap ini digunakan untuk menentukan apakah berita memiliki indikator yang berkaitan dengan Sport atau Finance. Sistem juga memiliki strong keyword untuk masing-masing domain sebagai indikator yang lebih kuat. Apabila tidak ditemukan kata kunci Sport maupun Finance yang mencukupi, artikel akan dikategorikan sebagai Other. Dengan adanya tahap ini, sistem tidak memaksakan berita yang berada di luar cakupan Sport dan Finance untuk masuk ke salah satu kategori tersebut.

Jika artikel terdeteksi memiliki domain Sport atau Finance, vektor dokumen kemudian diproses menggunakan MinMaxScaler untuk menyesuaikan skala nilai fitur. Setelah itu, vektor yang telah diskalakan dimasukkan ke dalam model Gaussian Naive Bayes untuk menentukan kategori berita. Gaussian Naive Bayes menghasilkan prediksi kelas serta probabilitas dari masing-masing kelas. Probabilitas tertinggi digunakan sebagai confidence score yang menunjukkan tingkat keyakinan model terhadap hasil klasifikasi.

Tahap terakhir adalah menampilkan hasil klasifikasi pada halaman web. Sistem menampilkan informasi berupa URL berita, judul artikel, isi berita, token hasil preprocessing, kategori hasil klasifikasi, confidence score, probabilitas masing-masing kelas, kata kunci domain yang ditemukan, vocabulary coverage, serta parameter Word2Vec yang digunakan. Dengan demikian, keseluruhan proses sistem dimulai dari input URL, web scraping, preprocessing, ekstraksi fitur menggunakan Word2Vec Skip-Gram, deteksi domain, normalisasi menggunakan MinMaxScaler, klasifikasi menggunakan Gaussian Naive Bayes, hingga menghasilkan informasi hasil klasifikasi pada antarmuka aplikasi.

## Link Deploy

[https://uts-23-022.vercel.app/](https://uts-23-022.vercel.app/)